# Laboratorio 7 — Árboles de Decisión y Bosques Aleatorios
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 30 de septiembre de 2026 · Sesión 7 de 17
**Bloques:** 6-8 (11:40-13:40, 105 min efectivos + recreo) · Lab. Central 114
**Unidad 2:** Algoritmos supervisados y no supervisados

**Objetivo:** dibujar un árbol de verdad y leerlo como el diagrama de preguntas de la mañana,
reproducir la curva de sobre/subajuste de la Clase 5 variando `max_depth` en vez de $k$, entrenar
un Random Forest y leer sus `feature_importances_`, y cerrar comparando los cuatro modelos
supervisados vistos hasta ahora — KNN, regresión logística, árbol y bosque — sobre el mismo dataset.

> **De dónde venimos:** en el Lab 1 un árbol de decisión sacó score **1.0**… evaluándose sobre sus
> propios datos de entrenamiento. Desde la Clase 5 sabemos que eso es tramposo. Hoy, con Gini y
> ganancia de información de la mañana, abrimos ese árbol del todo — y después entrenamos cien.

---
## Parte 1 · Un árbol de verdad, dibujado y controlado

### 1. Setup y primer árbol, sin restricciones

La carga y el split ya están dados (Wine, el mismo del Lab 1).

**Antes de correr, responde:** ¿esperas que el score de train sea 1.0 otra vez, aunque ahora hay
un test set separado?

> *(tu respuesta)*

**Tu tarea:** crea `tree_full = DecisionTreeClassifier(random_state=42)` (sin restringir nada),
entrénalo con el train e imprime su score en train y en test.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

**Responde aquí:** ¿cuánto vale la brecha train/test? ¿Qué firma de la Clase 5 es esa?

> *(tu respuesta)*

### 2. Dibujar el árbol

`max_depth=2` **dentro de `plot_tree`** solo corta el dibujo para que sea legible: el árbol
entrenado sigue siendo el completo de arriba.

**Tu tarea:** descomenta y corre el dibujo.

In [ ]:
# plt.figure(figsize=(20, 10))
# plot_tree(
#     tree_full, max_depth=2, feature_names=X.columns, class_names=wine.target_names,
#     filled=True, rounded=True, fontsize=9
# )
# plt.show()

**Responde aquí:**
1. Lee la **raíz** en voz alta: ¿qué pregunta hace (`feature <= umbral`)? ¿Cuántas muestras de
   cada clase llegan ahí (`value`)?
2. Verifica **a mano** el Gini de la raíz con la fórmula de la mañana, $G = 1 - \sum_k p_k^2$.
   ¿Coincide con el que imprime el dibujo?
3. Compara la primera pregunta del árbol con el ranking de `SelectKBest` del Lab 4:

| # | feature | F (Lab 4) |
|---|---|---|
| 1 | `flavanoids` | 151.6 |
| 2 | `od280/od315_of_diluted_wines` | 141.7 |
| 3 | `proline` | 123.7 |
| 4 | `color_intensity` | 92.1 |
| 5 | `alcohol` | 86.9 |

   ¿La feature de la raíz es la número 1 del ranking? Si no, ¿por qué podría no serlo?

> *(tus respuestas)*

**Tu tarea (camino de una muestra):** imprime `color_intensity`, `flavanoids` y `proline` de la
primera muestra del test (`X_test.iloc[0]`), sigue su camino por el dibujo a mano, y compara tu
predicción con `tree_full.predict(X_test.iloc[[0]])`.

### 3. Controlando la profundidad, una restricción a mano

**Tu tarea:** entrena `tree_shallow = DecisionTreeClassifier(max_depth=2, random_state=42)` e
imprime train y test.

**Completa la tabla:**

| modelo | train | test |
|---|---|---|
| `tree_full` (sin restricción) | | |
| `tree_shallow` (`max_depth=2`) | | |

**Responde aquí:** ¿bajó el score de train? ¿Y el de test? ¿Restringir el árbol **arregló** algo
en este caso, o se pasó para el otro lado? Usa las palabras *sobreajuste* y *subajuste*.

> *(tu respuesta)*

---
## ☕ Recreo

---
## Parte 2 · La curva en U, otra vez — ahora con `max_depth`

### 4. Barrido de profundidades: la curva de sobre/subajuste

**Tu tarea:**
1. Para `d` en `range(1, 15)`, entrena `DecisionTreeClassifier(max_depth=d, random_state=42)` y
   guarda los scores de train y test en `train_scores` y `test_scores`.
2. Grafica ambas curvas contra `max_depth` (con leyenda y ejes rotulados).

In [ ]:
depths = range(1, 15)
train_scores, test_scores = [], []

**Responde aquí:**
1. ¿En qué `max_depth` toca techo el score de test?
2. ¿Por qué las curvas quedan **completamente planas** desde cierta profundidad? *(pista: imprime
   `tree_full.get_depth()`)*
3. ¿Se ve la U de la curva de $k$ de KNN de la Clase 5?

> *(tus respuestas)*

**Tu tarea (con validación cruzada):** un solo split de 54 muestras de test es ruidoso (Clase 5).
Repite el barrido con `cross_validate(..., X, y, cv=5, return_train_score=True)` y grafica el
promedio de `train_score` y `test_score` por profundidad.

In [ ]:
from sklearn.model_selection import cross_validate

**Responde aquí:** ¿aparece ahora la U? ¿Dónde está el punto dulce? Compara con la curva de
$k$ de KNN: ¿qué hace el mismo papel en cada modelo?

> *(tu respuesta)*

### 5. Random Forest: el mismo dataset, muchos árboles

**Tu tarea:** entrena `forest = RandomForestClassifier(n_estimators=100, random_state=42)` e
imprime train y test.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

**Responde aquí:**
1. ¿El bosque también memoriza el train? ¿Y en test, cómo le va contra el mejor árbol de la
   sección 4?
2. ¿Tuviste que buscar un `max_depth` óptimo? ¿Qué dos fuentes de aleatoriedad (de la mañana)
   hacen que cada uno de los 100 árboles sea distinto?

> *(tus respuestas)*

### 6. Importancia de características

**Tu tarea:** arma una `pd.Series` con `forest.feature_importances_` indexada por `X.columns`,
ordenada de mayor a menor, y muéstrala.

In [ ]:
import pandas as pd

**Responde aquí:** compara el top-5 con el ranking de `SelectKBest` (sección 2) y con la
raíz del árbol dibujado. ¿Coinciden? ¿Por qué no tendrían por qué coincidir exactamente?

> *(tu respuesta)*

---
## Parte 3 · Los cuatro modelos, cara a cara

### 7. KNN, logística, árbol y bosque — el mismo dataset, la misma vara

**Antes de correr, responde:** ¿cuál esperas que gane? ¿Importa mucho la diferencia con un
dataset tan limpio como Wine?

> *(tu respuesta)*

**Tu tarea:**
1. Arma un diccionario `modelos` con: `StandardScaler` + `KNeighborsClassifier()`,
   `StandardScaler` + `LogisticRegression(max_iter=1000)` (ambos en `Pipeline`),
   `DecisionTreeClassifier(max_depth=4, random_state=42)` y
   `RandomForestClassifier(n_estimators=100, random_state=42)`.
2. Recórrelo con `cross_val_score(modelo, X, y, cv=5)` e imprime media ± desviación.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

**Responde aquí:**
1. ¿Por qué KNN y logística llevan `StandardScaler` y árbol y bosque no? *(gancho a la Clase 3)*
2. ¿Qué modelo tiene la desviación más grande? Relaciónalo con la palabra *inestable* de la mañana.
3. Si dos modelos empatan en score, ¿cuál elegirías y por qué?

> *(tus respuestas)*

### 8. Ejercicio propuesto (no evaluado)

Repite la comparación de la sección 7 sobre **Breast Cancer** (Clases 4 y 6). ¿Cambia el orden de
los cuatro modelos? ¿De qué depende la elección de modelo: de una tabla fija de "mejores
algoritmos" o de las características del dataset?

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)
X_c, y_c = cancer.data, cancer.target

---
**Moraleja del lab:** hoy se cerró el último "modelo usado sin explicar" del curso — el árbol del
Lab 1 — y apareció la primera técnica *ensemble*: combinar modelos simples e inestables para
conseguir uno robusto.

**Próxima clase:** SVM (máquinas de vectores de soporte) — otro modelo con frontera explícita, pero
con una idea geométrica distinta a árboles y regresión logística.